# IRIS full evaluation — PlotQA + FigureQA (half-split by ACCOUNT)

Evaluates the **existing IRIS pipeline (Algorithm 2)** — Gemma 3 4B and
Qwen3.5-4B each propose two sub-queries, an `all-mpnet-base-v2` vote picks
or merges them (with the fallback-exclusion rule), then the full **RoSE
pipeline runs on Qwen2.5-VL-7B** with the chosen sub-queries inserted as
steps — on:

- **PlotQA**: 500 questions per category (*structural*, *data retrieval*,
  *reasoning*), 1,500 total, using PlotQA's own per-question `template`
  labels (`structural`, `data_retrieval`, and the reasoning templates
  `min_max`/`arithmetic`/`comparison`/`compound`). Questions and golds are
  the official train `qa_pairs_V1` strings (full precision); the cauldron
  stream supplies only the chart images, with a runtime spot-check that
  cauldron row order equals official `image_index`.
- **FigureQA**: 100 questions per official question template
  (`question_id` 0–14), 1,500 total, from **validation1** (answers public,
  zero overlap with the train-split pilot by construction).

The 3,000-question sample is drawn once with a fixed seed (identical in
both notebooks) and split in half: **account A** runs 250 per PlotQA
category + 50 per FigureQA template, **account B** the other half. The
only difference between `iris_eval_account_A.ipynb` and
`iris_eval_account_B.ipynb` is the `ACCOUNT` line in the config cell.
The questions used by the earlier 250-question routed pilot runs are
excluded (PlotQA: the exact pilot picks are re-derived and skipped;
FigureQA: the pilot used the train split, this run uses validation1).

**Pipeline is untouched**: same code, prompts, hyperparameters and
`scoring.py` as `decomp_alg2_chartqapro.ipynb`, including its
`FAST_MODE = True` preset. This notebook only adds sampling, splitting,
checkpointing and reporting.

**Run order**: 5 rounds of 300 questions (50 per PlotQA category + 10 per
FigureQA template), PlotQA and FigureQA alternating inside a round, so an
early stop still leaves near-balanced data. Each dataset keeps its **own
RoSE experience pool** (as in the pilots — pools are never mixed across
datasets).

**Checkpointing**: every finished question is appended to
`results/iris_eval/iris_eval_results_<ACCOUNT>.jsonl` immediately; errors
are logged and skipped (retried on resume). To chain sessions, attach a
finished version's output (Add Input → Your Work → this notebook) or a
dataset holding the JSONL and set `RESULTS_INPUT_DIR` — answered
questions are skipped.

**Runtime + quota.** FAST mode ≈ 2.5–3.5 min/question on a T4 x2 →
one round (300 q) ≈ 12–18 h, the full 1,500 ≈ 60–90 h per account:
**6–8 budgeted sessions, i.e. 2–3 weeks of the 30 h/week quota.**
`MAX_HOURS = 11` stops cleanly at the end of the current question so a
headless *Save & Run All* version always publishes its outputs inside
Kaggle's 12 h session cap. A timing report prints after the first 100
questions of a session.

**Prerequisites**
1. Kaggle secret `HF_TOKEN` attached to this notebook (Gemma 3 is gated).
2. GPU: T4 x2 (Settings → Accelerator), Internet ON.
3. The `feature/iris-frontend` branch pushed to GitHub (this notebook clones it).


In [ ]:
# ── config ───────────────────────────────────────────────────────
ACCOUNT = "B"       # "A" or "B" — THE ONLY DIFFERENCE between the two notebooks

SEED      = 20261111      # fixed, IDENTICAL in both notebooks: draws the
                          # full 3,000-question sample, then splits it
MAX_HOURS = 11.0          # stop cleanly at the end of the current question
                          # (Kaggle kills the session at 12 h; a killed
                          # batch run publishes nothing)
FAST_MODE = True          # the decomp_alg2 notebook's speed preset —
                          # keep it the SAME across every chained session
DRY_RUN   = False         # True = answer only the first 4 pending questions
QUANT     = "nf4"         # same quantization as the pilot + decomp runs

REPO_URL  = "https://github.com/ktahsinr/RoSE.git"
REPO_REF  = "feature/iris-frontend"   # the branch with the IRIS Algorithm 2 code

# Chaining sessions: attach a finished version's output (Add Input → Your
# Work → this notebook) or a Kaggle dataset holding the results JSONL and
# point at the folder here; /kaggle/input/*/ is also searched automatically.
RESULTS_INPUT_DIR = None    # e.g. "/kaggle/input/<slug>/results/iris_eval"

# ── sampling design (do not change mid-run) ─────────────────────
ROUNDS                = 5
PLOTQA_PER_CATEGORY   = 500   # full sample; this account runs half
FIGUREQA_PER_TEMPLATE = 100   # full sample; this account runs half
N_PILOT               = 250   # questions used by each routed pilot run
PLOTQA_CHART_CAP      = 6000  # sampling universe: first N charts in the
                              # cauldron stream (deterministic prefix, same
                              # family as the pilots' first-250 window)

# PlotQA labels: official train qa_pairs_V1 (image_index, qid, template,
# question_string, answer) — mirror pinned to an exact revision so the
# sample is reproducible forever.
PLOTQA_MIRROR     = "Abd223653/PlotQA_V1"
PLOTQA_MIRROR_REV = "6d2e0107c9930d42f4ddaeb7df5006cfc31e8205"

# FigureQA: the official Microsoft release, validation1 (20k images,
# ~265k QAs, answers included, question_id 0-14 = the 15 templates).
FIGUREQA_URL = ("https://download.microsoft.com/download/c/3/1/"
                "c315c9d8-8239-487e-a895-2d3ff805b508/"
                "figureqa-validation1-v1.tar.gz")

import os, time
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # Kaggle can't render the widgets
T_START = globals().get("T_START", time.time())    # budget clock survives cell re-runs
from pathlib import Path
import sys
try:
    sys.stdout.reconfigure(line_buffering=True)    # live logs in Kaggle's pane
except (AttributeError, ValueError):
    pass

assert ACCOUNT in ("A", "B"), "ACCOUNT must be 'A' or 'B'"

ON_KAGGLE   = Path("/kaggle").exists()
WORK        = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
SCRATCH     = Path("/kaggle/temp")    if ON_KAGGLE else WORK / "scratch"
CODE_DIR    = SCRATCH / "RoSE"
IMAGES_DIR  = SCRATCH / "images"                   # plotqa/ and figureqa/ below
RESULTS_DIR = WORK / "results" / "iris_eval"       # published with the notebook

JSONL_FILE    = RESULTS_DIR / f"iris_eval_results_{ACCOUNT}.jsonl"
SPLIT_FILE    = RESULTS_DIR / "iris_eval_split.json"
MANIFEST_FILE = RESULTS_DIR / f"iris_eval_manifest_{ACCOUNT}.json"
REPORT_FILE   = RESULTS_DIR / f"iris_eval_report_{ACCOUNT}.csv"
META_FILE     = RESULTS_DIR / f"iris_eval_meta_{ACCOUNT}.json"

for d in (SCRATCH, IMAGES_DIR / "plotqa", IMAGES_DIR / "figureqa", RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# PlotQA template -> the paper's three categories
PLOTQA_REASONING = {"min_max", "arithmetic", "comparison", "compound"}
PLOTQA_CATEGORIES = ["structural", "data_retrieval", "reasoning"]
def plotqa_category(template):
    t = str(template).strip()
    if t == "structural":     return "structural"
    if t == "data_retrieval": return "data_retrieval"
    if t in PLOTQA_REASONING: return "reasoning"
    raise ValueError(f"unknown PlotQA template label: {template!r}")

# FigureQA question_id -> template name (docs/question_id_map.txt in the
# official repo; 2/12 and 3/13 repeat across categorical/line figures)
FIGUREQA_TEMPLATES = {
    0: "min", 1: "max", 2: "less", 3: "greater", 4: "low-median",
    5: "high-median", 6: "min-auc", 7: "max-auc", 8: "smoothest",
    9: "roughest", 10: "global-min", 11: "global-max", 12: "less-line",
    13: "greater-line", 14: "intersect",
}
def figureqa_type(question_id):
    return f"{int(question_id):02d}-{FIGUREQA_TEMPLATES[int(question_id)]}"

print(f"account={ACCOUNT}  seed={SEED}  rounds={ROUNDS}  fast={FAST_MODE}  "
      f"dry_run={DRY_RUN}  quant={QUANT}  max_hours={MAX_HOURS}")


In [ ]:
# ── code + pinned libraries (same as decomp_alg2_chartqapro) ─────
import subprocess
if not CODE_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF,
                    REPO_URL, str(CODE_DIR)], check=True)
else:
    # a kernel restart keeps /kaggle/temp — refresh so fixes pushed to the
    # branch mid-session are picked up instead of running stale code
    subprocess.run(["git", "-C", str(CODE_DIR), "fetch", "--depth", "1",
                    "origin", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(CODE_DIR), "reset", "--hard",
                    "FETCH_HEAD"], check=True)
print("repo @", subprocess.run(["git", "-C", str(CODE_DIR), "rev-parse", "--short", "HEAD"],
                               capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(CODE_DIR / "requirements-pinned.txt")], check=True)

# The sweep pin (transformers==4.51.3) supports Gemma 3 and Qwen2.5-VL
# but NOT Qwen3.5 (model_type "qwen3_5" -> ValueError at load time). Probe
# for it in a FRESH interpreter — before transformers is imported in this
# kernel, so an upgrade can never leave a stale module behind — and
# upgrade only if needed. The run's meta JSON records the exact version
# used either way, so the report stays honest.
def _knows_qwen35():
    return subprocess.run(
        [sys.executable, "-c",
         "from transformers.models.auto.configuration_auto import "
         "CONFIG_MAPPING; import sys; "
         "sys.exit(0 if 'qwen3_5' in CONFIG_MAPPING else 1)"],
    ).returncode == 0

if not _knows_qwen35():
    if "transformers" in sys.modules:
        raise SystemExit(
            "transformers needs an upgrade for Qwen3.5 but is already "
            "imported in this kernel — restart the kernel and Run All.")
    print("pinned transformers lacks the qwen3_5 architecture — "
          "upgrading to the latest release…")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "transformers"], check=True)
    if not _knows_qwen35():
        print("latest release still lacks qwen3_5 — installing from source…")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             "git+https://github.com/huggingface/transformers.git"],
            check=True)
        if not _knows_qwen35():
            raise SystemExit(
                "no available transformers build recognizes qwen3_5 — "
                "check https://huggingface.co/Qwen/Qwen3.5-4B for the "
                "version it requires")

if str(CODE_DIR / "chartqapro") not in sys.path:
    sys.path.insert(0, str(CODE_DIR / "chartqapro"))
import transformers
print("transformers:", transformers.__version__)


In [ ]:
# ── HuggingFace auth + gated-access check (all three models) ─────
# Every question touches all three models, so access to every one is
# verified up front — better a failure here than hours in.
NO_TOKEN_HELP = (
    "→ No HF token reached this session:\n"
    "  1. hf.co → Settings → Access Tokens → create a READ token\n"
    "     (use the SAME account that accepted the Gemma 3 license)\n"
    "  2. Kaggle → Add-ons → Secrets → add it with label HF_TOKEN\n"
    "  3. TICK THE CHECKBOX attaching the secret to THIS notebook —\n"
    "     the attachment is per-notebook and defaults to off\n"
    "  then re-run this cell.\n")

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print(f"could not read Kaggle secret HF_TOKEN: {e}")

if token:
    from huggingface_hub import login
    login(token=token)
    print("HF login ok")
elif ON_KAGGLE:
    raise SystemExit("\nNO HF TOKEN.\n" + NO_TOKEN_HELP)

from decompose.models import check_gated_access
check_gated_access()


In [ ]:
# ── PlotQA labels: official train qa_pairs_V1 (pinned mirror) ────
# Streams the label table (no images) for the first PLOTQA_CHART_CAP
# charts: image_index, qid (74 fine templates), template (the 6 coarse
# labels the paper groups into structural / data retrieval / reasoning),
# question_string and the official gold answer. The mirror is the full
# 5,733,893-row train qa_pairs_V1, sorted by image_index; the revision
# pin makes the sample reproducible forever.
import json
from datasets import load_dataset

LABELS_FILE = SCRATCH / f"plotqa_labels_{PLOTQA_CHART_CAP}.json"

if LABELS_FILE.exists():
    plotqa_labels = {int(k): v for k, v in
                     json.loads(LABELS_FILE.read_text()).items()}
    print(f"label table already built → {LABELS_FILE} "
          f"({len(plotqa_labels)} charts)")
else:
    mirror = load_dataset(PLOTQA_MIRROR, revision=PLOTQA_MIRROR_REV,
                          split="train", streaming=True)
    plotqa_labels, last = {}, -1
    for row in mirror:
        idx = int(row["image_index"])
        if idx < last:
            raise SystemExit(
                f"mirror rows are not sorted by image_index ({idx} after "
                f"{last}) — the early-stop assumption is wrong; do not "
                "trust this label table")
        last = idx
        if idx >= PLOTQA_CHART_CAP:
            break
        plotqa_labels.setdefault(idx, []).append({
            "q":        str(row["question_string"]).strip(),
            "answer":   str(row["answer"]).strip(),
            "template": str(row["template"]).strip(),
            "qid":      str(row["qid"]).strip(),
        })
    LABELS_FILE.write_text(json.dumps(plotqa_labels))
    print(f"label table: {len(plotqa_labels)} charts, "
          f"{sum(len(v) for v in plotqa_labels.values())} QAs → {LABELS_FILE}")

from collections import Counter
cat_counts = Counter(plotqa_category(qa["template"])
                     for qas in plotqa_labels.values() for qa in qas)
print("QAs per category in the sampling universe:", dict(cat_counts))


In [ ]:
# ── PlotQA images + cauldron texts (first PLOTQA_CHART_CAP charts) ─
# One pass over the `plotqa` subset of HuggingFaceM4/the_cauldron (the
# pilots' source): saves every chart image and the per-chart question
# list. The texts reproduce the pilot's exact picks (for exclusion) and
# verify that cauldron row order == official image_index (the label join).
import json

TEXTS_FILE = SCRATCH / f"plotqa_cauldron_texts_{PLOTQA_CHART_CAP}.json"
PQ_IMG_DIR = IMAGES_DIR / "plotqa"

if TEXTS_FILE.exists():
    cauldron_texts = {int(k): v for k, v in
                      json.loads(TEXTS_FILE.read_text()).items()}
    print(f"cauldron texts already built → {TEXTS_FILE} "
          f"({len(cauldron_texts)} charts)")
else:
    from datasets import load_dataset
    ds = load_dataset("HuggingFaceM4/the_cauldron", "plotqa",
                      split="train", streaming=True)
    cauldron_texts = {}
    for row_idx, row in enumerate(ds):
        if row_idx >= PLOTQA_CHART_CAP:
            break
        qs = []
        for qa in (row["texts"] or []):
            q = qa["user"].split("\n")[0].strip()   # drop "Make the answer very short." etc.
            a = qa["assistant"].strip().rstrip(".")
            qs.append([q, a])
        cauldron_texts[row_idx] = qs
        if row["images"]:
            path = PQ_IMG_DIR / f"{row_idx:06d}.png"
            if not path.exists():
                row["images"][0].convert("RGB").save(path, format="PNG")
        if (row_idx + 1) % 1000 == 0:
            print(f"  …{row_idx + 1}/{PLOTQA_CHART_CAP} charts", flush=True)
    TEXTS_FILE.write_text(json.dumps(cauldron_texts))
    print(f"{len(cauldron_texts)} charts (images + texts) → {PQ_IMG_DIR}")

# ── verify the label join: cauldron row i must hold the same questions
# as official image_index i. The cauldron reformats text (no space before
# '?', rounded answers with a trailing '.', extra QA variants), so the
# comparison is on NORMALIZED question text. A failure here means the
# alignment assumption is wrong and NOTHING should be run.
import re

def _norm_q(q):
    q = re.sub(r"\s+", " ", str(q).strip().lower())
    q = re.sub(r"\s+([?!.,;:])", r"\1", q)
    return q.rstrip(" ?.!").strip()

# The cauldron holds only part of each chart's official QAs and some
# extra variants, and GENERIC questions ("Does the graph contain grids?")
# repeat across charts — so the test is DISCRIMINATIVE: the official
# questions of image i must overlap cauldron row i clearly better than
# the NEIGHBOURING row. (Measured on live data: ~66% self vs ~25%
# neighbour.)
def _overlap(mirror_qs, row):
    c = {_norm_q(q) for q, _ in cauldron_texts.get(row, [])}
    return len(mirror_qs & c) / max(len(mirror_qs), 1)

n_checked = n_matched = 0
for idx in range(0, min(PLOTQA_CHART_CAP, len(cauldron_texts)) - 1, 97):
    mirror_qs = {_norm_q(qa["q"]) for qa in plotqa_labels.get(idx, [])}
    if not mirror_qs:
        continue
    n_checked += 1
    r_self, r_next = _overlap(mirror_qs, idx), _overlap(mirror_qs, idx + 1)
    if r_self >= 0.4 and r_self >= r_next + 0.15:
        n_matched += 1
if n_checked == 0 or n_matched / n_checked < 0.95:
    raise SystemExit(
        f"ALIGNMENT CHECK FAILED: only {n_matched}/{n_checked} spot-checked "
        "charts have question sets that match their own cauldron row better "
        "than the neighbouring row. Cauldron row order != image_index — "
        "the PlotQA image join is invalid. Stop and investigate.")
print(f"alignment OK: {n_matched}/{n_checked} spot-checked charts match "
      "their own cauldron row")

# ── reproduce the routed pilot's picks (first N_PILOT usable charts,
# one short non-yes/no answer each) so those exact questions are excluded.
MAX_GOLD_CHARS = 30
pilot_used, usable = set(), 0
for idx in sorted(cauldron_texts):
    pick = None
    for q, a in cauldron_texts[idx]:
        if not q or not a or len(a) > MAX_GOLD_CHARS:
            continue
        if a.lower() in ("yes", "no"):
            continue
        pick = q
        break
    if pick is not None:
        usable += 1
        pilot_used.add((idx, pick))
    if usable >= N_PILOT:
        break
print(f"pilot exclusion set: {len(pilot_used)} (chart, question) pairs "
      f"(charts 0–{max(i for i, _ in pilot_used)})")


In [ ]:
# ── FigureQA validation1: official Microsoft release ─────────────
# 403 MB tar: qa_pairs.json (question_id 0-14 = the 15 official
# templates, answers 0/1) + 20k PNGs. The pilot ran on the TRAIN split
# (via the_cauldron), so validation1 overlaps it by construction in
# exactly zero questions.
import json, tarfile, urllib.request

FQ_TAR = SCRATCH / "figureqa-validation1-v1.tar.gz"
FQ_QA  = SCRATCH / "figureqa_validation1_qa_pairs.json"

if not FQ_TAR.exists() or FQ_TAR.stat().st_size < 400_000_000:
    print("downloading figureqa-validation1-v1.tar.gz (403 MB)…", flush=True)
    urllib.request.urlretrieve(FIGUREQA_URL, FQ_TAR)
print(f"tar: {FQ_TAR.stat().st_size / 1e6:.1f} MB")

if not FQ_QA.exists():
    with tarfile.open(FQ_TAR, "r:gz") as tf:
        member = next(m for m in tf if m.name.endswith("qa_pairs.json"))
        FQ_QA.write_bytes(tf.extractfile(member).read())

figureqa_qa = json.loads(FQ_QA.read_text())["qa_pairs"]
print(f"validation1: {len(figureqa_qa)} QAs over "
      f"{len({r['image_index'] for r in figureqa_qa})} images")
from collections import Counter
print("QAs per template:",
      dict(sorted(Counter(r["question_id"] for r in figureqa_qa).items())))


In [ ]:
# ── draw the full sample (seed-fixed, identical in both notebooks),
#    split A/B, assign rounds, save the manifests ─────────────────
import json, random

HALF_PQ = PLOTQA_PER_CATEGORY // 2      # 250 per category per account
HALF_FQ = FIGUREQA_PER_TEMPLATE // 2    # 50 per template per account
PQ_PER_ROUND = HALF_PQ // ROUNDS        # 50
FQ_PER_ROUND = HALF_FQ // ROUNDS        # 10

# PlotQA candidates per category, in deterministic mirror order. The
# QUESTION POSED and the GOLD are the official qa_pairs_V1 strings (the
# dataset's own, full precision — the cauldron rounds answers to 2
# decimals, which distorts small-magnitude golds past scoring.py's 5%
# numeric tolerance); the cauldron supplies only the chart image, whose
# identity the alignment spot-check above validated. The pilot's
# questions are excluded by normalized question text within the chart.
pilot_used_norm = {(i, _norm_q(q)) for i, q in pilot_used}
pq_cands = {c: [] for c in PLOTQA_CATEGORIES}
seen = set()
for idx in sorted(plotqa_labels):
    for pos, qa in enumerate(plotqa_labels[idx]):
        key = (idx, _norm_q(qa["q"]))
        if key in seen or key in pilot_used_norm:
            continue
        seen.add(key)
        pq_cands[plotqa_category(qa["template"])].append({
            "id":        f"plotqa-{idx:06d}-q{pos:02d}",
            "dataset":   "plotqa",
            "eval_type": plotqa_category(qa["template"]),
            "template":  qa["template"],
            "qid":       qa["qid"],
            "question":  qa["q"],
            "answer":    qa["answer"],
            "image":     str(IMAGES_DIR / "plotqa" / f"{idx:06d}.png"),
            "image_index": idx,
        })
print("PlotQA candidates:", {c: len(v) for c, v in pq_cands.items()})

# FigureQA candidates per template, in official qa_pairs order
fq_cands = {t: [] for t in range(15)}
for q_index, rec in enumerate(figureqa_qa):
    fq_cands[int(rec["question_id"])].append({
        "id":        f"figureqa-v1-{q_index:06d}",
        "dataset":   "figureqa",
        "eval_type": figureqa_type(rec["question_id"]),
        "question":  str(rec["question_string"]).strip(),
        "answer":    "Yes" if int(rec["answer"]) == 1 else "No",
        "image":     str(IMAGES_DIR / "figureqa" / f"{rec['image_index']}.png"),
        "image_index": int(rec["image_index"]),
    })

# seeded draw per stratum → first half A, second half B
split = {"A": {}, "B": {}}   # account -> stratum -> list of records
for cat in PLOTQA_CATEGORIES:
    rng = random.Random(f"{SEED}-plotqa-{cat}")
    assert len(pq_cands[cat]) >= PLOTQA_PER_CATEGORY, \
        f"only {len(pq_cands[cat])} candidates for {cat} — raise PLOTQA_CHART_CAP"
    drawn = rng.sample(pq_cands[cat], PLOTQA_PER_CATEGORY)
    split["A"][f"plotqa/{cat}"] = drawn[:HALF_PQ]
    split["B"][f"plotqa/{cat}"] = drawn[HALF_PQ:]
for tpl in range(15):
    rng = random.Random(f"{SEED}-figureqa-{tpl}")
    assert len(fq_cands[tpl]) >= FIGUREQA_PER_TEMPLATE
    drawn = rng.sample(fq_cands[tpl], FIGUREQA_PER_TEMPLATE)
    split["A"][f"figureqa/{tpl:02d}"] = drawn[:HALF_FQ]
    split["B"][f"figureqa/{tpl:02d}"] = drawn[HALF_FQ:]

# sanity: A and B share no question id; strata are full-size
ids_a = {r["id"] for s in split["A"].values() for r in s}
ids_b = {r["id"] for s in split["B"].values() for r in s}
assert not ids_a & ids_b, "A/B overlap!"
assert len(ids_a) == len(ids_b) == 3 * HALF_PQ + 15 * HALF_FQ
# sanity: no pilot question slipped in (normalized-text comparison)
assert not any((r["image_index"], _norm_q(r["question"])) in pilot_used_norm
               for s in split["A"].values() for r in s
               if r["dataset"] == "plotqa"), "pilot leak in A"
assert not any((r["image_index"], _norm_q(r["question"])) in pilot_used_norm
               for s in split["B"].values() for r in s
               if r["dataset"] == "plotqa"), "pilot leak in B"

# round assignment + run order: inside a round, PlotQA (cycling the 3
# categories) and FigureQA (cycling the 15 templates) strictly alternate,
# so ANY prefix of the run is near-balanced across datasets and strata.
def run_order(account):
    ordered = []
    for rnd in range(ROUNDS):
        pq = [split[account][f"plotqa/{cat}"][rnd * PQ_PER_ROUND + k]
              for k in range(PQ_PER_ROUND) for cat in PLOTQA_CATEGORIES]
        fq = [split[account][f"figureqa/{tpl:02d}"][rnd * FQ_PER_ROUND + k]
              for k in range(FQ_PER_ROUND) for tpl in range(15)]
        assert len(pq) == len(fq) == 3 * PQ_PER_ROUND
        block = [x for pair in zip(pq, fq) for x in pair]
        for r in block:
            r = dict(r)
            r["round"] = rnd + 1
            ordered.append(r)
    return ordered

RUN_LIST = run_order(ACCOUNT)
print(f"account {ACCOUNT}: {len(RUN_LIST)} questions in {ROUNDS} rounds "
      f"of {len(RUN_LIST) // ROUNDS}")

# save the split (both accounts — identical from either notebook) and
# this account's full manifest, so the split can be checked offline
SPLIT_FILE.write_text(json.dumps({
    "seed": SEED, "rounds": ROUNDS,
    "plotqa_per_category": PLOTQA_PER_CATEGORY,
    "figureqa_per_template": FIGUREQA_PER_TEMPLATE,
    "plotqa_chart_cap": PLOTQA_CHART_CAP,
    "plotqa_mirror": {"repo": PLOTQA_MIRROR, "revision": PLOTQA_MIRROR_REV},
    "figureqa_source": FIGUREQA_URL,
    "pilot_excluded_plotqa": sorted([i, q] for i, q in pilot_used),
    "A": sorted(ids_a), "B": sorted(ids_b),
}, indent=1))
MANIFEST_FILE.write_text(json.dumps(RUN_LIST, indent=1))
print(f"split ids → {SPLIT_FILE.name}   manifest → {MANIFEST_FILE.name}")

# extract this account's FigureQA images from the tar (one pass)
import tarfile
needed = {r["image_index"] for r in RUN_LIST if r["dataset"] == "figureqa"}
missing = {i for i in needed
           if not (IMAGES_DIR / "figureqa" / f"{i}.png").exists()}
if missing:
    print(f"extracting {len(missing)} FigureQA images…", flush=True)
    with tarfile.open(FQ_TAR, "r:gz") as tf:
        for m in tf:
            if not m.name.endswith(".png"):
                continue
            stem = Path(m.name).stem
            if stem.isdigit() and int(stem) in missing:
                (IMAGES_DIR / "figureqa" / f"{int(stem)}.png").write_bytes(
                    tf.extractfile(m).read())
                missing.discard(int(stem))
                if not missing:
                    break
assert not missing, f"images not found in tar: {sorted(missing)[:5]}"
for r in RUN_LIST:                      # every image must exist before we run
    assert Path(r["image"]).exists(), f"missing image for {r['id']}"
print("all images in place")


In [ ]:
# ── RoSE config + the FAST_MODE preset (same as decomp_alg2) ─────
# Every value is SET explicitly (not asserted), so re-running this cell
# is always safe.
if "CODE_DIR" not in globals():
    raise SystemExit("run the config cell first — after a kernel restart, "
                     "use Run All so the cells execute in order")
if not (CODE_DIR / "chartqapro").exists():
    raise SystemExit("repo not cloned yet — run the clone cell first")
if str(CODE_DIR / "chartqapro") not in sys.path:
    sys.path.insert(0, str(CODE_DIR / "chartqapro"))

import rose_chartqapro as rose
import decompose.decomp_runner as decomp_runner

assert (rose.K_DEMONSTRATIONS, rose.LAMBDA, rose.TEMPERATURE) == (3, 1.2, 0.8), \
    "rose_chartqapro defaults drifted from the sweep config"
assert not rose.PAPER_FAITHFUL
rose.IMAGES_DIR = str(IMAGES_DIR)

rose.M_PATHS = 3 if FAST_MODE else 5
rose.EXTENSIONS["two_stage_reasoning"] = not FAST_MODE
rose.EXTENSIONS["visual_hybrid_retrieval"] = not FAST_MODE
rose.MAX_PIXELS = (1024 if FAST_MODE else 1280) * 28 * 28
decomp_runner.DECOMP_MAX_TOKENS = 128 if FAST_MODE else 256

print(f"{'FAST' if FAST_MODE else 'SPEC-EXACT'} config:  "
      f"m={rose.M_PATHS}  k={rose.K_DEMONSTRATIONS}  λ={rose.LAMBDA}  "
      f"T={rose.TEMPERATURE}  max_pixels={rose.MAX_PIXELS}  "
      f"decomp_tokens={decomp_runner.DECOMP_MAX_TOKENS}")
for name, on in rose.active_extensions().items():
    print(f"   {'✓' if on else '·'} {name}")

# The one embedder for everything: RoSE demo retrieval and the
# decomposition vote. Retries transient HF Hub 500s.
from sentence_transformers import SentenceTransformer

def _load_embedder(name, attempts=6):
    try:                       # a complete local cache needs no Hub at all
        return SentenceTransformer(name, local_files_only=True)
    except TypeError:
        pass
    except Exception:
        pass                   # not cached yet — download below
    last = None
    for i in range(1, attempts + 1):
        try:
            return SentenceTransformer(name)
        except Exception as e:
            last = e
            wait = min(45 * i, 240)
            print(f"embedder download failed ({type(e).__name__}) — "
                  f"retry {i}/{attempts} in {wait}s")
            time.sleep(wait)
    raise SystemExit(
        f"could not load {name} after {attempts} attempts — the HF Hub is "
        f"having problems (last error: {last}). Re-run this cell in a few "
        "minutes; everything already downloaded is cached.")

if "embedder" not in globals():
    embedder = _load_embedder(rose.EMBED_MODEL)
embed_fn = lambda texts: embedder.encode(list(texts), convert_to_numpy=True)
print(f"embedder: {rose.EMBED_MODEL}")


In [ ]:
# ── resume: load every existing result, rebuild the two pools ────
# Results live in iris_eval_results_<ACCOUNT>.jsonl (one JSON object per
# line, appended right after each question). Sources, in order: the
# working file, RESULTS_INPUT_DIR, any attached input. Error rows are
# dropped (and so retried); the working JSONL is rewritten as the merged,
# deduplicated history so it is always cumulative.
import json

def _jsonl_rows(path):
    rows = []
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if line:
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"  ! skipping corrupt line in {path}")
    return rows

sources = []
if JSONL_FILE.exists():
    sources.append(JSONL_FILE)
if RESULTS_INPUT_DIR:
    p = Path(RESULTS_INPUT_DIR) / JSONL_FILE.name
    if p.exists():
        sources.append(p)
if ON_KAGGLE:
    sources += sorted(Path("/kaggle/input").glob(f"*/{JSONL_FILE.name}"))
    sources += sorted(Path("/kaggle/input").glob(
        f"*/results/iris_eval/{JSONL_FILE.name}"))

results, done = [], set()
for src in sources:
    for r in _jsonl_rows(src):
        if r.get("id") in done:
            continue
        if r.get("method") == "error":
            continue                      # dropped → retried this session
        done.add(r["id"])
        results.append(r)
if sources:
    print("sources:", *[str(s) for s in sources], sep="\n  ")
err_ids = {r.get("id") for src in sources for r in _jsonl_rows(src)
           if r.get("method") == "error"} - done
if err_ids:
    print(f"↩  {len(err_ids)} error row(s) dropped — will retry")
print(f"↩  {len(done)}/{len(RUN_LIST)} already answered")

# rewrite the working JSONL as the merged history (append-only afterwards)
with JSONL_FILE.open("w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

# one experience pool PER DATASET (as in the pilots — never mixed),
# rebuilt in original answer order, exactly as run_decomp resumes
pools = {"plotqa": rose.ExperiencePool(embedder),
         "figureqa": rose.ExperiencePool(embedder)}
for r in results:
    pools[r["dataset"]].add(
        question=r["question"],
        rationale=r.get("best_rationale") or r.get("prediction", ""),
        answer=r.get("prediction", ""),
        qtype="factoid",
        uncertainty=r.get("uncertainty", 0.5),
        complexity=r.get("complexity", 1.0),
        clip_emb=None)
print(f"pools rebuilt: plotqa={pools['plotqa'].size()}  "
      f"figureqa={pools['figureqa'].size()}")

pending = [r for r in RUN_LIST if r["id"] not in done]
if DRY_RUN:
    pending = pending[:4]
print(f"pending: {len(pending)}" + ("  (DRY RUN)" if DRY_RUN else ""))


In [ ]:
# ── run Algorithm 2 (GPU) — JSONL checkpoint after EVERY question ─
# All three 4-bit VLMs are loaded ONCE and stay resident (≈13 GB — needs
# the T4 x2). Identical per-question pipeline to run_decomp's Algorithm 2
# branch (_run_alg2 is the pipeline; this loop only adds the JSONL
# append, the per-dataset pools, rounds, the 100-question timing report
# and the MAX_HOURS budget). A failed question is logged and skipped.
import time, traceback
from collections import Counter
import csv as _csv
from decompose.decomp_runner import (load_all_models, _run_alg2, _params_b,
                                     _rescore)
from decompose.models import env_versions
import scoring

PARAMS_B = _params_b(2)

if "loaded" not in globals():
    loaded, revisions = load_all_models(QUANT)

clip_model = clip_processor = None
if rose.ext("visual_hybrid_retrieval"):
    clip_model, clip_processor = rose.load_clip()


def report_rows(rows):
    """accuracy + count per dataset and per type, rescored with scoring.py"""
    out, keys = [], []
    for r in rows:
        if r.get("method") == "error":
            continue
        k = (r["dataset"], r["eval_type"])
        if k not in keys:
            keys.append(k)
    for ds, tp in sorted(keys):
        sub = [r for r in rows if r.get("method") != "error"
               and r["dataset"] == ds and r["eval_type"] == tp]
        n_ok = sum(scoring.is_correct(r.get("prediction", ""),
                                      r.get("gold", ""), "factoid")
                   for r in sub)
        out.append({"dataset": ds, "type": tp, "n": len(sub), "correct": n_ok,
                    "accuracy_pct": round(n_ok / max(len(sub), 1) * 100, 2)})
    for ds in ("plotqa", "figureqa", "OVERALL"):
        sub = [r for r in rows if r.get("method") != "error"
               and (ds == "OVERALL" or r["dataset"] == ds)]
        if not sub:
            continue
        n_ok = sum(scoring.is_correct(r.get("prediction", ""),
                                      r.get("gold", ""), "factoid")
                   for r in sub)
        out.append({"dataset": ds, "type": "ALL", "n": len(sub),
                    "correct": n_ok,
                    "accuracy_pct": round(n_ok / max(len(sub), 1) * 100, 2)})
    return out


def write_report(rows, path):
    table = report_rows(rows)
    with Path(path).open("w", newline="", encoding="utf-8") as f:
        w = _csv.DictWriter(f, fieldnames=["dataset", "type", "n", "correct",
                                           "accuracy_pct"])
        w.writeheader()
        w.writerows(table)
    return table


def print_report(table, title):
    print(f"\n── {title} " + "─" * max(1, 50 - len(title)))
    for row in table:
        print(f"  {row['dataset']:<9} {row['type']:<16} n={row['n']:>4}  "
              f"{row['accuracy_pct']:6.2f}%")


session_times, out_of_time = [], False
budget_s = MAX_HOURS * 3600 if MAX_HOURS else None

# rounds already complete (earlier sessions): re-write their CSVs so this
# version's output is self-contained, but don't re-print them
rounds_reported = set()
for rnd in range(1, ROUNDS + 1):
    if all(x["id"] in done for x in RUN_LIST if x["round"] == rnd):
        rounds_reported.add(rnd)
        write_report([x for x in results if x["round"] <= rnd],
                     RESULTS_DIR / f"iris_eval_report_{ACCOUNT}_round{rnd}.csv")
if rounds_reported:
    print(f"rounds already complete: {sorted(rounds_reported)}")

print(f"\n━━ IRIS eval, account {ACCOUNT}: {len(pending)} pending, "
      f"models resident: {', '.join(loaded)} ━━")
for rec in pending:
    if budget_s is not None and time.time() - T_START > budget_s:
        print(f"⏱  MAX_HOURS ({MAX_HOURS:.1f} h) reached — stopping cleanly "
              f"with {len(done)}/{len(RUN_LIST)} answered; chain the next "
              "session from this notebook's output")
        out_of_time = True
        break

    sample = {"id": rec["id"], "question": rec["question"], "choices": None,
              "answer": rec["answer"], "question_type": "factoid",
              "image": rec["image"]}
    t0 = time.time()
    try:
        r = _run_alg2(sample, loaded, pools[rec["dataset"]],
                      clip_model, clip_processor, embed_fn)
        r["params_b_used"] = PARAMS_B
    except Exception as exc:
        traceback.print_exc()
        r = {"id": rec["id"], "question": rec["question"],
             "question_type": "factoid", "ground_truth": rec["answer"],
             "prediction": "ERROR", "is_correct": False, "algorithm": 2,
             "error": str(exc), "method": "error"}
    seconds = round(time.time() - t0, 2)
    r["latency_s"] = seconds

    row = {
        "id":         rec["id"],
        "dataset":    rec["dataset"],
        "eval_type":  rec["eval_type"],
        "round":      rec["round"],
        "account":    ACCOUNT,
        "question":   rec["question"],
        "gold":       rec["answer"],
        "prediction": r.get("prediction", ""),
        "correct":    _rescore(r),
        "seconds":    seconds,
        # pipeline detail, for analysis
        "prediction_norm": r.get("prediction_norm", ""),
        "vote_rule":   r.get("vote_rule", ""),
        "sub_queries": r.get("sub_queries", []),
        "method":      r.get("method", ""),
        "uncertainty": r.get("uncertainty", ""),
        "complexity":  r.get("complexity", ""),
        "agreement":   r.get("agreement", ""),
        "n_demos":     r.get("n_demos", ""),
        "pool_size":   r.get("pool_size", ""),
        "best_rationale": r.get("best_rationale", ""),
        "error":       r.get("error", ""),
        "ts":          time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    with JSONL_FILE.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
    if row["method"] != "error":
        results.append(row)
        done.add(rec["id"])
        session_times.append(seconds)

    mark = "✗✓"[bool(row["correct"])] if row["method"] != "error" else "!"
    print(f"[{len(done):04d}/{len(RUN_LIST)}] {mark} r{rec['round']} "
          f"{rec['dataset']:<8} {rec['eval_type']:<16} {seconds:6.1f}s "
          f"pool={pools[rec['dataset']].size():<3} "
          f"rule={r.get('vote_rule') or '—':<16} "
          f"pred={str(r.get('prediction'))[:24]}", flush=True)

    # timing guard: after the first 100 questions of this session
    if len(session_times) == 100:
        avg = sum(session_times) / len(session_times)
        rem = len(RUN_LIST) - len(done)
        sessions = (f" (≈ {rem * avg / 3600 / MAX_HOURS:.1f} more "
                    f"{MAX_HOURS:.0f}h sessions)") if MAX_HOURS else ""
        print(f"\n⏲  TIMING after 100 questions: {avg:.1f} s/question — "
              f"projected {rem * avg / 3600:.1f} h for the remaining "
              f"{rem} questions of this account{sessions}\n")

    # end-of-round report, once per completed round
    rnd = rec["round"]
    if rnd not in rounds_reported and all(
            x["id"] in done for x in RUN_LIST if x["round"] == rnd):
        rounds_reported.add(rnd)
        rows_r = [x for x in results if x["round"] <= rnd]
        table = write_report(
            rows_r, RESULTS_DIR / f"iris_eval_report_{ACCOUNT}_round{rnd}.csv")
        print_report(table, f"through round {rnd} ({len(rows_r)} questions)")

# cumulative report + meta, written on EVERY stop (budget or complete)
table = write_report(results, REPORT_FILE)
print_report(table, f"cumulative, account {ACCOUNT}")
if session_times:
    avg = sum(session_times) / len(session_times)
    rem = len(RUN_LIST) - len(done)
    print(f"\nthis session: {len(session_times)} questions, "
          f"{avg:.1f} s/question; {rem} remaining "
          f"(≈ {rem * avg / 3600:.1f} h)")
META_FILE.write_text(json.dumps({
    "account": ACCOUNT, "algorithm": 2, "seed": SEED,
    "n_answered": len(done), "n_total": len(RUN_LIST),
    "complete": len(done) >= len(RUN_LIST) and not out_of_time,
    "fast_mode": FAST_MODE, "quant": QUANT, "params_b_used": PARAMS_B,
    "vote_rules": dict(Counter(r["vote_rule"] for r in results
                               if r.get("vote_rule"))),
    "revisions": {m: (revisions or {}).get(m) for m in loaded},
    "rose_config": {
        "m_paths": rose.M_PATHS, "k_demonstrations": rose.K_DEMONSTRATIONS,
        "lambda": rose.LAMBDA, "temperature": rose.TEMPERATURE,
        "max_new_tokens": rose.MAX_NEW_TOKENS, "max_pixels": rose.MAX_PIXELS,
        "extensions": rose.active_extensions(),
    },
    "env": env_versions(),
    "finished_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}, indent=2))
print(f"\n✓ results → {JSONL_FILE.name}   report → {REPORT_FILE.name}   "
      f"meta → {META_FILE.name}")


In [ ]:
# ── final report: accuracy + count per dataset and per type ──────
# Reads the cumulative JSONL (so it also works standalone after a
# restart) and prints/saves the same scoring.py-rescored table.
import json

rows = [json.loads(l) for l in JSONL_FILE.read_text().splitlines() if l.strip()]
rows = [r for r in rows if r.get("method") != "error"]
table = write_report(rows, REPORT_FILE)
print_report(table, f"FINAL — account {ACCOUNT} ({len(rows)} answered)")
by_round = {}
for r in rows:
    by_round.setdefault(r["round"], []).append(r)
print("\nper round:", {k: len(v) for k, v in sorted(by_round.items())})
print(f"\nsaved → {REPORT_FILE}")
print(f"download from the Output tab: {JSONL_FILE.name}, {REPORT_FILE.name}, "
      f"{SPLIT_FILE.name}, {MANIFEST_FILE.name}, {META_FILE.name} "
      f"and the per-round CSVs")


In [ ]:
# ── merge accounts A and B into one combined table ───────────────
# Run this in either notebook once both JSONL files are available —
# attach the other account's published output (or upload its JSONL as a
# dataset) so both files are visible under /kaggle/input or /kaggle/working.
import json

found = {}
roots = [WORK, RESULTS_DIR] + (
    [p for p in Path("/kaggle/input").glob("*")] if ON_KAGGLE else [])
for acct in ("A", "B"):
    name = f"iris_eval_results_{acct}.jsonl"
    for root in roots:
        for p in sorted(root.rglob(name)):
            found.setdefault(acct, p)
print("found:", {k: str(v) for k, v in found.items()})

if len(found) < 2:
    print("→ only one account's results are visible; attach the other "
          "account's output to merge.")
else:
    merged, ids = [], set()
    for acct, path in sorted(found.items()):
        for line in path.read_text().splitlines():
            if not line.strip():
                continue
            r = json.loads(line)
            if r.get("method") == "error":
                continue
            assert r["id"] not in ids, f"duplicate question across accounts: {r['id']}"
            ids.add(r["id"])
            merged.append(r)
    combined_csv = RESULTS_DIR / "iris_eval_report_combined.csv"
    table = write_report(merged, combined_csv)
    print_report(table, f"COMBINED A+B ({len(merged)} questions)")
    print(f"\nsaved → {combined_csv}")


## After the session

**Headless chaining (Save & Run All):** with `MAX_HOURS` set, each
committed version runs up to the budget, stops cleanly at the end of the
current question, and PUBLISHES its outputs. If
`iris_eval_meta_<ACCOUNT>.json` says `"complete": false`:
Add Input → Your Work → this notebook (attaches the latest version's
output — the JSONL under `/kaggle/input/...` is found automatically, or
set `RESULTS_INPUT_DIR`), re-pin the input to the newest version, and
Save & Run All again. Answered questions are skipped, errored ones
retried. **Keep `FAST_MODE` and the sampling config identical across
chained sessions.**

1. Download from the Output pane: `iris_eval_results_<ACCOUNT>.jsonl`,
   `iris_eval_report_<ACCOUNT>.csv` (+ per-round CSVs),
   `iris_eval_split.json`, `iris_eval_manifest_<ACCOUNT>.json`,
   `iris_eval_meta_<ACCOUNT>.json`.
2. The split can be audited offline: `iris_eval_split.json` holds the
   seed, the pinned data sources, the excluded pilot questions and the
   full id lists for both accounts (identical no matter which notebook
   wrote it).
3. When BOTH accounts are done, run the merge cell above with the other
   account's output attached — it asserts the halves are disjoint and
   writes `iris_eval_report_combined.csv`.
4. Quota bookkeeping: ≈ 60–90 GPU-h per account at FAST mode — plan
   6–8 budgeted sessions inside the 30 h/week limit (≈ 2–3 weeks).
